# 04 - RAG con LangChain

Objetivo: recuperar contexto desde Qdrant y generar una respuesta final con Ollama.


In [1]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama, OllamaEmbeddings
from qdrant_client import QdrantClient

load_dotenv(Path("..").resolve() / ".env")

OLLAMA_PORT = os.getenv("OLLAMA_PORT", "11434")
QDRANT_PORT = os.getenv("QDRANT_PORT", "6333")
CHAT_MODEL = os.getenv("CHAT_MODEL", "llama3")
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "embeddinggemma")
COLLECTION_NAME = os.getenv("COLLECTION_NAME", "frasohome_docs")
TOP_K = int(os.getenv("TOP_K", "3"))

OLLAMA_BASE_URL = f"http://127.0.0.1:{OLLAMA_PORT}"
QDRANT_URL = f"http://127.0.0.1:{QDRANT_PORT}"

chat_model = ChatOllama(model=CHAT_MODEL, base_url=OLLAMA_BASE_URL, temperature=0)
embedding_model = OllamaEmbeddings(model=EMBEDDING_MODEL, base_url=OLLAMA_BASE_URL)
client = QdrantClient(url=QDRANT_URL)


In [2]:
prompt = ChatPromptTemplate.from_template(
    """
Responde solo con la informacion del contexto.
Si el contexto no es suficiente, di claramente que no lo sabes.

Pregunta: {question}

Contexto:
{context}
    """.strip()
)
chain = prompt | chat_model | StrOutputParser()

def retrieve(question: str, top_k: int = TOP_K):
    query_vector = embedding_model.embed_query(question)
    return client.query_points(collection_name=COLLECTION_NAME, query=query_vector, limit=top_k).points

def build_context(hits):
    blocks = []
    for hit in hits:
        source = hit.payload.get("source", "desconocido")
        chunk_id = hit.payload.get("chunk_id", "?")
        text = hit.payload.get("text", "")
        blocks.append(f"Fuente: {source} | Chunk: {chunk_id}\n{text}")
    return "\n\n".join(blocks)

def ask_rag(question: str, top_k: int = TOP_K):
    hits = retrieve(question, top_k=top_k)
    context = build_context(hits)
    answer = chain.invoke({"question": question, "context": context})
    return answer, hits, context


In [3]:
question = "Puedo devolver una mesa ya montada si solo he cambiado de opinion?"
answer, hits, context = ask_rag(question)

print("RESPUESTA")
print(answer)
print("\nFUENTES")
for hit in hits:
    print({
        "score": round(hit.score, 4),
        "source": hit.payload.get("source"),
        "chunk_id": hit.payload.get("chunk_id"),
    })


RESPUESTA
No, no puedes devolver una mesa ya montada si solo has cambiado de opinión. Según el contexto, los productos ya montados no se aceptan como devolución por desistimiento si el motivo es simplemente un cambio de opinión, color o medida percibida por el cliente tras el montaje.

FUENTES
{'score': 0.6364, 'source': 'C:\\Repos\\rag-local-lab\\docs\\politica_devoluciones.md', 'chunk_id': 5}
{'score': 0.5839, 'source': 'C:\\Repos\\rag-local-lab\\docs\\politica_devoluciones.md', 'chunk_id': 4}
{'score': 0.5227, 'source': 'C:\\Repos\\rag-local-lab\\docs\\politica_devoluciones.md', 'chunk_id': 7}


In [4]:
evaluation_path = Path("..").resolve() / "evaluation" / "questions.csv"
df = pd.read_csv(evaluation_path)
df = df.head(5).copy()

answers = []
for question in df["question"]:
    answer, hits, _ = ask_rag(question)
    answers.append({
        "question": question,
        "answer": answer,
        "top_source": hits[0].payload.get("source") if hits else None,
    })

pd.DataFrame(answers)


,question,answer,top_source
0,¿Puedo devolver una mesa ya montada si simplem...,"No, no puedes devolver una mesa ya montada si ...",C:\Repos\rag-local-lab\docs\politica_devolucio...
1,¿Qué herramientas necesito para montar la mesa...,"Para montar la mesa Oslo, necesitas:\n\n* La l...",C:\Repos\rag-local-lab\docs\manual_mesa_oslo.md
2,¿Cuánto tiempo tengo para comunicar un daño de...,Tienes 72 horas para comunicar un daño de tran...,C:\Repos\rag-local-lab\docs\politica_devolucio...
3,¿Puedo devolver un producto personalizado?,"No, no puedes devolver un producto personalizado.",C:\Repos\rag-local-lab\docs\politica_devolucio...
4,¿Qué hago si faltan tornillos en la caja?,"Si faltan tornillos en la caja, no fuerces el ...",C:\Repos\rag-local-lab\docs\manual_mesa_oslo.md


Pruebas sugeridas:

1. Cambia `TOP_K` y compara la respuesta.
2. Haz una pregunta que no este en los documentos.
3. Repite la indexacion del notebook anterior tras cambiar `CHUNK_SIZE`.
